In [ ]:
import json
import argparse

import matplotlib.pyplot as plt
from matplotlib.patches import Patch

In [ ]:
MASTER_LABEL = "Master"
LOAD_COLOR = "#7F8C8D"
MERGE_COLOR = "#5B7DB1"
ENCODE_COLOR = "#2E8B57"
OVERHEAD_COLOR = "#BFE3CC"
FAIL_COLORS = {
    "failed": "#D9534F",
    "worker_lost": "#F0AD4E",
}
FAIL_NAMES = {
    "failed": "Segmen gagal (retry)",
    "worker_lost": "Worker tidak terjangkau",
}

In [ ]:
def load_log(path):
    with open(path, "r", encoding = "utf-8") as f:
        return json.load(f)

def plot_gantt(log, output = None, show = True):
    timeline = [r for r in log["timeline"] if r["end"] is not None]
    workers = log.get("workers") or sorted({r["worker_id"] for r in timeline})
    summary = log.get("summary", {})
    worker_summary = summary.get("workers", {})
    merge = log.get("merge")

    load_time = log.get("load_video_time") or 0.0
    shift = load_time

    has_merge = bool(merge and merge.get("duration"))
    rows = ([MASTER_LABEL] if (load_time > 0 or has_merge) else []) + list(workers)
    y_of = {name: index for index, name in enumerate(rows)}

    makespan = summary.get("makespan") or max((r["end"] for r in timeline), default = 0.0)
    encode_end = makespan + shift
    x_max = max(encode_end, (merge["end"] + shift) if has_merge else 0.0)

    fig, ax = plt.subplots(figsize = (max(8, min(24, x_max / 4)), 1.2 + 0.8 * len(rows)))
    label_box = dict(boxstyle = "round,pad=0.1", fc = "white", ec = "none", alpha = 0.7)

    if load_time > 0:
        ax.barh(y_of[MASTER_LABEL], load_time, left = 0, height = 0.6, color = LOAD_COLOR, edgecolor = "black")

        if load_time >= 0.03 * x_max:
            ax.text(load_time / 2, y_of[MASTER_LABEL], "Load", ha = "center", va = "center", fontsize = 8, bbox = label_box)

    if has_merge:
        merge_color = "#D9534F" if merge.get("error") else MERGE_COLOR
        ax.barh(y_of[MASTER_LABEL], merge["duration"], left = merge["start"] + shift, height = 0.6, color = merge_color, edgecolor = "black")

        if merge["duration"] >= 0.03 * x_max:
            ax.text(merge["start"] + shift + merge["duration"] / 2, y_of[MASTER_LABEL], "Merge", ha = "center", va = "center", fontsize = 8, bbox = label_box)

    has_overhead = False

    for r in timeline:
        left = r["start"] + shift
        wall = r["end"] - r["start"]
        y = y_of[r["worker_id"]]
        encode = r.get("encoding_time")

        if r["status"] == "completed" and encode:
            encode = min(encode, wall)
            ax.barh(y, encode, left = left, height = 0.6, color = ENCODE_COLOR, edgecolor = "black")

            if wall - encode > 0:
                has_overhead = True
                ax.barh(y, wall - encode, left = left + encode, height = 0.6, color = OVERHEAD_COLOR, edgecolor = "black", hatch = "//")
        else:
            ax.barh(y, wall, left = left, height = 0.6, color = FAIL_COLORS.get(r["status"], "#999999"), edgecolor = "black")

        label = f"S{r['segment']}" if r["attempt"] == 1 else f"S{r['segment']} ({r['attempt']})"

        if wall >= 0.03 * x_max:
            ax.text(left + wall / 2, y, label, ha = "center", va = "center", fontsize = 8, bbox = label_box)

    ax.axvline(encode_end, color = "black", linestyle = "--", linewidth = 0.8)

    ticklabels = []

    for name in rows:
        if name in worker_summary:
            ticklabels.append(f"{name}\nutil {worker_summary[name]['utilization'] * 100:.0f}%")
        else:
            ticklabels.append(name)

    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels(ticklabels)
    ax.invert_yaxis()
    ax.set_xlim(0, x_max * 1.02)
    ax.set_xlabel("Waktu (detik)")
    ax.grid(axis = "x", linestyle = ":", alpha = 0.5)
    ax.set_axisbelow(True)

    title = f"Job {log.get('job_id')} | scheduler: {log.get('scheduler')} | makespan encode: {makespan:.2f}s"

    if load_time > 0:
        title += f" | load video: {load_time:.2f}s"

    if has_merge:
        title += f" | merge: {merge['duration']:.2f}s"

    ax.set_title(title, fontsize = 10)

    legend = []

    if load_time > 0:
        legend.append(Patch(facecolor = LOAD_COLOR, edgecolor = "black", label = "Load video"))

    if any(r["status"] == "completed" for r in timeline):
        legend.append(Patch(facecolor = ENCODE_COLOR, edgecolor = "black", label = "Encode"))

    if has_overhead:
        legend.append(Patch(facecolor = OVERHEAD_COLOR, edgecolor = "black", hatch = "//", label = "Overhead (jaringan dll.)"))

    for status, color in FAIL_COLORS.items():
        if any(r["status"] == status for r in timeline):
            legend.append(Patch(facecolor = color, edgecolor = "black", label = FAIL_NAMES[status]))

    if has_merge:
        legend.append(Patch(facecolor = MERGE_COLOR, edgecolor = "black", label = "Merge segmen"))

    ax.legend(handles = legend, loc = "upper center", bbox_to_anchor = (0.5, -0.18), ncol = max(1, len(legend)), frameon = False)
    fig.tight_layout()

    if output:
        fig.savefig(output, dpi = 200, bbox_inches = "tight")

    if show:
        plt.show()

    return fig

In [ ]:
LOG_PATH = r""

In [ ]:
plot_gantt(load_log(LOG_PATH), output = None, show = False)